In [4]:
!python -m pip install gymnasium stable_baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 4.3 MB/s eta 0:00:0000:01


실제 장비와의 **통신 인터페이스(Modbus/MQTT 가상 예시)**, **시간대별 목표 온도 스케줄링**, 그리고 **성능 리포트 자동 생성** 기능을 모두 통합한 최종 마스터 코드입니다.

이 코드는 `REAL_MODE` 변수에 따라 **가상 시뮬레이션 학습**과 **실제 장비 제어**를 자유롭게 전환할 수 있도록 설계되었습니다.

#### **1. 통합 제어 및 리포트 시스템 코드**

In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from collections import deque
import time

# [설정] 실제 장비 연결 여부 (True: 장비 제어, False: 시뮬레이션 학습)
REAL_MODE = False 

In [2]:

class IntegratedDryerEnv(gym.Env):
    def __init__(self):
        super(IntegratedDryerEnv, self).__init__()
        # 액션: [히터 0:Off/1:On, 압력 0:유지/1:증가/2:감소]
        self.action_space = spaces.MultiDiscrete([2, 3])
        # 상태: [현재온도, 오차, 현재압력, 온도변화율]
        self.observation_space = spaces.Box(low=-150, high=150, shape=(4,), dtype=np.float32)
        
        # 1. 스케줄링 설정 (분 단위 또는 스텝 단위)
        # 0~100스텝: 50도, 101~200스텝: 80도, 201~300스텝: 60도
        self.schedule = {100: 50.0, 200: 80.0, 300: 60.0}
        self.history = deque([(0, 0.5)] * 5, maxlen=5) # 5스텝 열지연
        self.reset()

    def _get_target_temp(self):
        for limit, temp in sorted(self.schedule.items()):
            if self.steps <= limit: return temp
        return 60.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        print(f'IntegratedDryerEnv.reset.seed ={seed}')
        self.steps = 0
        self.pressure = 0.5
        self.current_temp = 30.0 if not REAL_MODE else self._read_real_sensor()
        self.target_temp = self._get_target_temp()
        return self._get_obs(), {}

    def _get_obs(self):
        diff = self.current_temp - self.target_temp
        return np.array([self.current_temp, diff, self.pressure, 0.0], dtype=np.float32)

    def _read_real_sensor(self):
        # [실제 장비 연동] Modbus/MQTT 등으로 온도 읽기
        # return modbus_client.read_input_registers(0x01)
        return 30.0 # 예시값

    def _send_real_command(self, h, p):
        # [실제 장비 연동] PLC 등에 제어 명령 송신
        # modbus_client.write_register(0x02, h)
        pass

    def step(self, actions):
        self.steps += 1
        print(f'IntegratedDryerEnv.step.steps ={self.steps}')
        self.target_temp = self._get_target_temp()
        self.history.append(actions)
        h_act, p_act = self.history[0] # 지연된 액션 적용

        if REAL_MODE:
            self._send_real_command(h_act, p_act)
            time.sleep(1) # 제어 주기 맞춤
            self.current_temp = self._read_real_sensor()
        else:
            # 시뮬레이션 물리 엔진
            if p_act == 1: self.pressure = min(1.0, self.pressure + 0.02)
            elif p_act == 2: self.pressure = max(0.0, self.pressure - 0.02)
            heat = 4.2 if h_act == 1 else 0.0
            cool = self.pressure * 6.0
            self.current_temp += (heat - cool - (self.current_temp-20)*0.03) + np.random.normal(0, 0.05)

        diff = abs(self.current_temp - self.target_temp)
        reward = 5.0 if diff < 0.5 else 1.0 - (diff * 0.2)
        
        terminated = self.steps >= 300
        return self._get_obs(), reward, terminated, False, {}



In [3]:

# [Step 2] 학습 및 저장 함수
def train_and_save(model_name="dryer_master_model"):
    env = IntegratedDryerEnv()
    model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0003)
    model.learn(total_timesteps=50000)
    model.save(model_name)
    return model


In [4]:

# [Step 3] 성능 리포트 생성 함수
def run_and_report(model_path):
    env = IntegratedDryerEnv()
    model = PPO.load(model_path)
    obs, _ = env.reset()
    
    logs = []
    for i in range(300):
        print(f'step [{i}]')
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, _, _ = env.step(action)
        logs.append({
            'Step': i, 'Target': env.target_temp, 'Current': obs[0], 
            'Error': abs(obs[1]), 'Pressure': env.pressure, 'Heater': action[0]
        })
    
    df = pd.DataFrame(logs)
    print(f"\n[성능 요약] MAE: {df['Error'].mean():.4f} | Max Error: {df['Error'].max():.4f}")
    
    # 시각화 리포트
    plt.figure(figsize=(15, 7))
    plt.plot(df['Target'], 'g--', label='Target Schedule')
    plt.plot(df['Current'], 'r-', label='AI Control Temp', linewidth=2)
    plt.bar(df.index, df['Heater']*5, alpha=0.2, color='orange', label='Heater On')
    plt.fill_between(df.index, 0, df['Pressure']*100, alpha=0.1, color='blue', label='Pressure %')
    plt.title("Dryer AI Control Performance Report (Lag & Schedule Applied)")
    plt.legend()
    plt.show()
    return df


In [5]:

# 실행
# 1. 학습 (최초 1회)
train_and_save("final_model")
print('completed to train and save model')
# 2. 리포트 생성 및 검증
# report_data = run_and_report("final_model")

IntegratedDryerEnv.reset.seed =None
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
IntegratedDryerEnv.reset.seed =None
IntegratedDryerEnv.step.steps =1
IntegratedDryerEnv.step.steps =2
IntegratedDryerEnv.step.steps =3
IntegratedDryerEnv.step.steps =4
IntegratedDryerEnv.step.steps =5
IntegratedDryerEnv.step.steps =6
IntegratedDryerEnv.step.steps =7
IntegratedDryerEnv.step.steps =8
IntegratedDryerEnv.step.steps =9
IntegratedDryerEnv.step.steps =10
IntegratedDryerEnv.step.steps =11
IntegratedDryerEnv.step.steps =12
IntegratedDryerEnv.step.steps =13
IntegratedDryerEnv.step.steps =14
IntegratedDryerEnv.step.steps =15
IntegratedDryerEnv.step.steps =16
IntegratedDryerEnv.step.steps =17
IntegratedDryerEnv.step.steps =18
IntegratedDryerEnv.step.steps =19
IntegratedDryerEnv.step.steps =20
IntegratedDryerEnv.step.steps =21
IntegratedDryerEnv.step.steps =22
IntegratedDryerEnv.step.steps =23
IntegratedDryerEnv.step.steps =24
IntegratedDryerEnv.step.ste